# 1.1 — Генерация синтетических данных

**Папка 1 «Подготовка данных», подноутбук 1.** Генерирует популяцию грунтов с полным
геотехническим набором свойств: гранулометрический состав (11 фракций), плотности,
влажность и пределы Аттерберга, пористость, органика, карбонатность, классификация по
ГОСТ (type_ground) и по PLAXIS, а также механические параметры, включая все входы
физической модели CRR (Dr, e, Ip, содержание мелких/глинистых частиц, σ′, OCR, K0, Vs1,
цементация, старение и др.) и общие характеристики (E, c, φ). По свойствам грунта строится
кривая сопротивления CRR(N) = β / N^(1 − α). Все рисунки и таблицы — на английском.

## Окружение

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_source_synthetic"   # источник синтетики (НЕ канонический demo_run)
MODELS_DIR = REPO_ROOT / "models"

from liquefaction_ai import generate_population, get_default_config, save_population_artifact, set_global_seed
from liquefaction_ai.constants import FEATURE_AXIS_LABELS_EN, LOAD_DISPLAY_NAMES_EN, SOIL_DISPLAY_NAMES_EN
from liquefaction_ai.viz import histogram_grid, lines

print("Repository root:", REPO_ROOT)

Repository root: /sessions/determined-cool-fermat/mnt/liquefaction-ai


## Конфигурация и генерация

In [3]:
config = get_default_config()
set_global_seed(config.seed)
import os
if os.environ.get("LIQ_QUICK"):
    from dataclasses import replace
    config = replace(config, n_scenarios=1200, benchmark_subset=800)
population = generate_population(config)
meta = population["meta"]
save_population_artifact(DATA_DIR, population, config)
print("Scenarios:", len(meta), "| meta columns:", meta.shape[1], "| saved to:", DATA_DIR)
display(pd.DataFrame({
    "Metric": ["Population size", "Liquefaction rate", "Mean N_liq (cycles)",
               "Median peak PPR", "Mean CRR15", "Soil types present"],
    "Value": [len(meta), round(float(meta["liq_label"].mean()), 4),
              round(float(meta["N_liq_true"].mean()), 1),
              round(float(meta["PPR_max_true"].median()), 4),
              round(float(meta["crr_ref"].mean()), 4),
              int(meta["type_ground"].nunique())],
}))

/sessions/determined-cool-fermat/mnt/liquefaction-ai/src/liquefaction_ai/data/synthetic.py:390: RuntimeWarning: overflow encountered in exp
  phi = np.log1p(np.exp(6.0 * (ratio - 0.92))) / 6.0


Scenarios: 1200 | meta columns: 94 | saved to: /sessions/determined-cool-fermat/mnt/liquefaction-ai/data/demo_source_synthetic


,Metric,Value
0,Population size,1200.0000
1,Liquefaction rate,0.5458
2,Mean N_liq (cycles),508.5000
3,Median peak PPR,0.5826
4,Mean CRR15,0.3318
5,Soil types present,9.0000


## Состав датасета по типам грунта (ГОСТ) и физическим параметрам

In [4]:
from liquefaction_ai.constants import SOIL_NAMES
counts = meta["soil_type"].map(SOIL_DISPLAY_NAMES_EN).value_counts()
display(counts.rename("N scenarios").to_frame())

overview_cols = ["D_r", "e", "fines_content", "I_p", "V_s", "crr_ref"]
histogram_grid(meta, columns=overview_cols,
               titles=[FEATURE_AXIS_LABELS_EN[c] for c in overview_cols],
               n_cols=3, title="Overview of key soil parameters and CRR15",
               save=SAVE_FIGS, fig_id="1_1_parameter_overview").show()

,N scenarios
soil_type,
Silty sand,270
Medium sand,204
Sandy loam,177
Fine sand,162
Loam,121
Coarse sand,112
Gravelly sand,73
Clay,59
Peat,22


[save_figure] PNG для '1_1_parameter_overview' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Артефакт с полным геотехническим набором сохранён в `data/demo_run/`. Дальше — **1.2
разведочный анализ**, **1.3 анализ параметров CRR**, **1.4 разбиение выборок**.